# Build ML model for analysis and prediction of DNA-encoded library data.

In [3]:
import os
import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINConv, global_mean_pool
from torch_geometric.data import Data, Batch
from torch_geometric.loader import DataLoader

from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.Chem import Descriptors, rdPartialCharges
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold

import itertools
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import pathlib
from pathlib import Path
from rdkit import RDLogger
from tqdm.notebook import tqdm

# Get the main logger
logger = RDLogger.logger()

# Set its level to ERROR so only serious messages show up
logger.setLevel(RDLogger.ERROR)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#verify Cuda is actually working

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (runtime):", torch.version.cuda)

print("Torch CUDA libraries loaded:")
for lib in torch.cuda._CudaBase.__module__.split(":"):
    print(lib)

Torch version: 2.8.0+cu128
CUDA available: True
CUDA version (runtime): 12.8
Torch CUDA libraries loaded:
torch.cuda


## Setup.

In [4]:
readout = 'kindel_del'

In [5]:
# define paths
HERE = Path(pathlib.Path.cwd())
DATA = HERE / f"data_{readout}"
DATA.mkdir(parents=True, exist_ok=True)

In [6]:
import s3fs

for dataset in ['ddr1', 'mapk14']: 
    fs = s3fs.S3FileSystem(anon=True)
    info = fs.info(f'kin-del-2024/data/{dataset}.parquet')
    print(f"File size for {dataset}: {info['size']/1e6:.2f} MB")

File size for ddr1: 6782.18 MB
File size for mapk14: 6911.55 MB


In [7]:
import pyarrow.parquet as pq

def read_first_n_rows(file_path, n_rows):
    parquet_file = pq.ParquetFile(file_path)
    
    num_row_groups = parquet_file.num_row_groups
    
    rows_read = 0
    result = []

    for i in range(num_row_groups):
        row_group = parquet_file.read_row_group(i)
        df = row_group.to_pandas()

        if rows_read + len(df) >= n_rows:
            result.append(df.head(n_rows - rows_read))
            break
        
        result.append(df)
        rows_read += len(df)

    final_df = pd.concat(result, ignore_index=True).head(n_rows)
    return final_df

In [8]:
def count_rows(file_path):
    parquet_file = pq.ParquetFile(file_path)
    total_rows = sum(parquet_file.metadata.row_group(i).num_rows
                     for i in range(parquet_file.num_row_groups))
    return total_rows

In [90]:
target = f"{DATA}/ddr1.parquet"

row_count = count_rows(target)
print(f"Total rows: {row_count}")

N = 10000  # Number of rows to load - for prototyping
df = read_first_n_rows(target, N)

Total rows: 67641617


## Preprocessing

In [91]:
import numpy as np
import pandas as pd

# Average triplicates
df['target_mean'] = df[['seq_target_1','seq_target_2','seq_target_3']].mean(axis=1)
df['control_mean'] = df[['seq_matrix_1','seq_matrix_2','seq_matrix_3']].mean(axis=1)

# Add a pseudocount to avoid division by zero
pseudocount = 1

# Compute log2 enrichment
df['log_enrichment'] = np.log2((df['target_mean'] + pseudocount) / (df['control_mean'] + pseudocount))

# Optional: normalize by library loading
df['log_enrichment_norm'] = np.log2((df['target_mean'] + pseudocount) / (df['seq_load'] + pseudocount))


In [92]:
# Define a function to compute Morgan fingerprints (a type of molecular fingerprint)
def compute_fingerprint(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol:
        return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits))
    else:
        return None

# Compute the fingerprints for each synthon and the final molecule
def get_fingerprints(df):
    df['fp_a'] = df['smiles_a'].apply(compute_fingerprint)
    df['fp_b'] = df['smiles_b'].apply(compute_fingerprint)
    df['fp_c'] = df['smiles_c'].apply(compute_fingerprint)
    df['fp_final'] = df['smiles'].apply(compute_fingerprint)
    
    # Drop rows where any fingerprint is missing
    df = df.dropna(subset=['fp_a', 'fp_b', 'fp_c', 'fp_final']).reset_index(drop=True)
    return df

# Apply to your DataFrame
df = get_fingerprints(df)

# Convert fingerprints to numpy arrays (bit vectors)
fingerprints = np.array([
    np.concatenate([row['fp_a'], row['fp_b'], row['fp_c'], row['fp_final']])
    for _, row in df.iterrows()
])
    
fingerprints = np.array(fingerprints)

[02:14:44] SMILES Parse Error: syntax error while parsing: CC(C)(C)OC(=O)N1C[C@@H](C[C@H]1C(O)=O)NC(=O)OCC1c2ccccc2-c2ccccc12,
[02:14:44] SMILES Parse Error: Failed parsing SMILES 'CC(C)(C)OC(=O)N1C[C@@H](C[C@H]1C(O)=O)NC(=O)OCC1c2ccccc2-c2ccccc12,' for input: 'CC(C)(C)OC(=O)N1C[C@@H](C[C@H]1C(O)=O)NC(=O)OCC1c2ccccc2-c2ccccc12,'
[02:14:44] SMILES Parse Error: syntax error while parsing: [H][C@@](CCC(O)=O)(NC(=O)OC(C)(C)C)C(=O)OC,
[02:14:44] SMILES Parse Error: Failed parsing SMILES '[H][C@@](CCC(O)=O)(NC(=O)OC(C)(C)C)C(=O)OC,' for input: '[H][C@@](CCC(O)=O)(NC(=O)OC(C)(C)C)C(=O)OC,'
[02:14:44] SMILES Parse Error: syntax error while parsing: CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc12)C(O)=O,
[02:14:44] SMILES Parse Error: Failed parsing SMILES 'CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc12)C(O)=O,' for input: 'CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc2-c2ccccc12)C(O)=O,'
[02:14:44] SMILES Parse Error: syntax error while parsing: CC(C)(C)OC(=O)NCC[C@@H](NC(=O)OCC1c2ccccc

In [93]:
# Remove molecules with failed SMILES
df = df[df['fp_final'].notnull()]

# Filter molecules with very low pre-population
min_load = 10
df = df[df['seq_load'] >= min_load]


In [94]:
df['fp_final_arr'] = df['smiles'].apply(compute_fingerprint)

# Stack fingerprints as features
X = np.stack(df['fp_final_arr'].values)
y = df['log_enrichment'].values

## Baseline RF model

In [95]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt


In [96]:
from sklearn.model_selection import train_test_split

# 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} molecules")
print(f"Test set: {X_test.shape[0]} molecules")


Training set: 5760 molecules
Test set: 1441 molecules


In [97]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error

rf = RandomForestRegressor(
    n_estimators=500,
    n_jobs=-1,
    random_state=42
)

rf.fit(X_train, y_train)


,n_estimators,500
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [ ]:
# from sklearn.metrics import mean_squared_error
# import numpy as np

# y_pred = rf.predict(X_test)

# mse = mean_squared_error(y_test, y_pred)
# rmse = np.sqrt(mse)

# print(f"R² on test set: {r2:.3f}")
# print(f"RMSE on test set: {rmse:.3f}")



## GNN model - final molecule.

In [98]:
from rdkit import Chem
import torch
from torch_geometric.data import Data

def mol_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    # Node features
    atom_features = []
    for atom in mol.GetAtoms():
        atom_features.append([
            atom.GetAtomicNum(),
            atom.GetDegree(),
            int(atom.GetIsAromatic()),
            atom.GetFormalCharge()
        ])
    x = torch.tensor(atom_features, dtype=torch.float)
    
    # Edge indices and edge features
    edge_index = []
    edge_attr = []
    for bond in mol.GetBonds():
        i = bond.GetBeginAtomIdx()
        j = bond.GetEndAtomIdx()
        edge_index.append([i, j])
        edge_index.append([j, i])
        edge_attr.append([bond.GetBondTypeAsDouble()])
        edge_attr.append([bond.GetBondTypeAsDouble()])
    
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


In [99]:
from torch_geometric.data import InMemoryDataset

class DELDataset(InMemoryDataset):
    def __init__(self, df, transform=None):
        super().__init__('.', transform)
        self.df = df
        self.data_list = [mol_to_graph(s) for s in df['smiles']]
        self.data_list = [d for d in self.data_list if d is not None]  # drop parse errors
        # Add target
        for i, d in enumerate(self.data_list):
            d.y = torch.tensor([df['log_enrichment'].iloc[i]], dtype=torch.float)
    
    def len(self):
        return len(self.data_list)
    
    def get(self, idx):
        return self.data_list[idx]


In [100]:
from torch_geometric.loader import DataLoader

dataset = DELDataset(df)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)


### Model spec

In [101]:
import torch.nn as nn
from torch_geometric.nn import GINConv, global_add_pool

class GNNRegressor(nn.Module):
    def __init__(self, hidden_dim=64, num_layers=3):
        super().__init__()
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        for i in range(num_layers):
            nn1 = nn.Sequential(
                nn.Linear(hidden_dim if i > 0 else 4, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim)
            )
            self.convs.append(GINConv(nn1))
            self.bns.append(nn.BatchNorm1d(hidden_dim))
        
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = torch.relu(x)
        x = global_add_pool(x, batch)
        out = self.fc(x)
        return out


In [102]:
from sklearn.metrics import r2_score, mean_squared_error

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GNNRegressor().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

for epoch in range(20):
    # -----------------------
    # Training loop
    model.train()
    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        y_pred = model(batch)
        loss = criterion(y_pred.view(-1), batch.y.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch.num_graphs
    
    avg_train_loss = total_loss / len(train_loader.dataset)
    
    # -----------------------
    # Evaluation loop
    model.eval()
    y_true, y_pred_list = [], []
    with torch.no_grad():
        for batch in train_loader:  # use test_loader if evaluating on test set
            batch = batch.to(device)
            y_pred_batch = model(batch)
            y_true.append(batch.y.view(-1).cpu())
            y_pred_list.append(y_pred_batch.view(-1).cpu())
    
    y_true = torch.cat(y_true).numpy()
    y_pred_list = torch.cat(y_pred_list).numpy()
    
    r2 = r2_score(y_true, y_pred_list)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_list))  # manual RMSE
    
    print(f"Epoch {epoch+1}: Train Loss={avg_train_loss:.4f}, R²={r2:.3f}, RMSE={rmse:.3f}")


Epoch 1: Train Loss=1.8311, R²=0.398, RMSE=1.017
Epoch 2: Train Loss=1.0175, R²=0.403, RMSE=1.014
Epoch 3: Train Loss=0.9125, R²=0.376, RMSE=1.036
Epoch 4: Train Loss=0.8708, R²=0.483, RMSE=0.942
Epoch 5: Train Loss=0.9072, R²=-0.043, RMSE=1.339
Epoch 6: Train Loss=0.8243, R²=0.524, RMSE=0.905
Epoch 7: Train Loss=0.7424, R²=-0.340, RMSE=1.518
Epoch 8: Train Loss=0.7740, R²=0.558, RMSE=0.872
Epoch 9: Train Loss=0.7277, R²=0.484, RMSE=0.942
Epoch 10: Train Loss=0.7614, R²=-0.141, RMSE=1.401
Epoch 11: Train Loss=0.7385, R²=0.538, RMSE=0.892
Epoch 12: Train Loss=0.6979, R²=0.408, RMSE=1.009
Epoch 13: Train Loss=0.6859, R²=0.614, RMSE=0.815
Epoch 14: Train Loss=0.7057, R²=0.358, RMSE=1.051
Epoch 15: Train Loss=0.6748, R²=0.325, RMSE=1.077
Epoch 16: Train Loss=0.6797, R²=0.275, RMSE=1.117
Epoch 17: Train Loss=0.6596, R²=0.605, RMSE=0.825
Epoch 18: Train Loss=0.7057, R²=0.602, RMSE=0.827
Epoch 19: Train Loss=0.6331, R²=0.559, RMSE=0.870
Epoch 20: Train Loss=0.6178, R²=0.419, RMSE=1.000
